# Desafio Técnico — Analista de Dados
## Integração com APIs: Feriados + Clima
**APIs utilizadas:**
- [Public Holiday API](https://date.nager.at/Api) — feriados nacionais do Brasil
- [Open-Meteo Historical Weather](https://open-meteo.com/) — clima histórico diário

In [ ]:
# !pip install requests pandas --quiet

import requests
import pandas as pd
from datetime import date

YEAR       = 2024
COUNTRY    = "BR"
LAT, LON   = -22.9068, -43.1729   # Rio de Janeiro
START_DATE = "2024-01-01"
END_DATE   = "2024-08-01"

print("✅ Configuração ok!")

In [ ]:
# ==============================================================
# FUNÇÕES AUXILIARES
# ==============================================================

def get_holidays(year: int, country: str) -> pd.DataFrame:
    """Busca feriados nacionais via Public Holiday API."""
    url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country}"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    df = pd.DataFrame(resp.json())
    df["date"]        = pd.to_datetime(df["date"])
    df["weekday"]     = df["date"].dt.day_name()
    df["weekday_num"] = df["date"].dt.dayofweek   # 0=segunda, 6=domingo
    df["month"]       = df["date"].dt.month
    return df


def get_weather(lat: float, lon: float, start: str, end: str) -> pd.DataFrame:
    """Busca dados climáticos históricos diários via Open-Meteo."""
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start, "end_date": end,
        "daily": ["temperature_2m_mean", "weathercode"],
        "timezone": "America/Sao_Paulo",
    }
    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()["daily"]
    df = pd.DataFrame(data)
    df["time"] = pd.to_datetime(df["time"])
    df = df.rename(columns={
        "time": "date",
        "temperature_2m_mean": "temp_mean",
        "weathercode": "weather_code",
    })
    df["month"] = df["date"].dt.month
    return df


# Tabela WMO Weather Code → descrição
# Ref: https://gist.github.com/stellasphere/9490c195ed2b53c707087c8c2db4ec0c
WMO_DESCRIPTIONS = {
    0: "Céu limpo",
    1: "Principalmente limpo",
    2: "Parcialmente nublado",
    3: "Totalmente nublado",
    45: "Nevoeiro", 48: "Nevoeiro com geada",
    51: "Garoa leve", 53: "Garoa moderada", 55: "Garoa intensa",
    61: "Chuva leve", 63: "Chuva moderada", 65: "Chuva intensa",
    71: "Neve leve", 73: "Neve moderada", 75: "Neve intensa", 77: "Granizo",
    80: "Pancadas de chuva leves", 81: "Pancadas de chuva moderadas", 82: "Pancadas de chuva intensas",
    85: "Neve em pancadas", 86: "Neve em pancadas intensas",
    95: "Trovoada", 96: "Trovoada com granizo", 99: "Trovoada intensa com granizo",
}

def wmo_desc(code) -> str:
    if pd.isna(code):
        return "Sem dados"
    return WMO_DESCRIPTIONS.get(int(code), f"Código {int(code)}")

# Códigos considerados ruins para ir à praia
# (totalmente nublado, chuva de qualquer tipo, tempestade, neve, nevoeiro)
BAD_WEATHER_CODES = {3, 45, 48, 51, 53, 55, 61, 63, 65, 71, 73, 75, 77,
                     80, 81, 82, 85, 86, 95, 96, 99}

def is_beach_day(row) -> bool:
    """True se o dia tem sol (não totalmente nublado/chuvoso) E temp >= 20°C."""
    if pd.isna(row["temp_mean"]) or pd.isna(row["weather_code"]):
        return None
    return (row["temp_mean"] >= 20) and (int(row["weather_code"]) not in BAD_WEATHER_CODES)


print("✅ Funções carregadas!")

In [ ]:
# ==============================================================
# BUSCAR DADOS DAS APIs
# ==============================================================
print("Buscando feriados...")
holidays_all = get_holidays(YEAR, COUNTRY)

print("Buscando dados climáticos...")
weather = get_weather(LAT, LON, START_DATE, END_DATE)

# Feriados dentro do período com dados de clima (jan–ago)
holidays_period = holidays_all[holidays_all["date"] <= pd.Timestamp(END_DATE)].copy()

print(f"✅ {len(holidays_all)} feriados em 2024 | {len(weather)} dias de clima (jan–ago)")

---
### Pergunta 1 — Quantos feriados há no Brasil em todo o ano de 2024?

In [ ]:
print(f"Total de feriados nacionais no Brasil em 2024: {len(holidays_all)}\n")
display(holidays_all[["date", "localName", "name", "weekday"]].reset_index(drop=True))

---
### Pergunta 2 — Qual mês de 2024 tem o maior número de feriados?

In [ ]:
by_month = (
    holidays_all.groupby("month")
    .size()
    .reset_index(name="qtd_feriados")
    .sort_values("qtd_feriados", ascending=False)
)
by_month["mes"] = by_month["month"].apply(lambda m: date(2024, m, 1).strftime("%B"))

top = by_month.iloc[0]
print(f"Mês com mais feriados: {top['mes']} ({top['qtd_feriados']} feriados)\n")
display(by_month[["mes", "qtd_feriados"]].reset_index(drop=True))

---
### Pergunta 3 — Quantos feriados em 2024 caem em dias úteis (segunda a sexta)?

In [ ]:
weekday_h = holidays_all[holidays_all["weekday_num"] < 5].copy()
print(f"Feriados em dias úteis em 2024: {len(weekday_h)}\n")
display(weekday_h[["date", "localName", "weekday"]].reset_index(drop=True))

---
### Pergunta 4 — Temperatura média em cada mês (jan–ago 2024)?

In [ ]:
monthly_temp = (
    weather.groupby("month")["temp_mean"]
    .mean()
    .round(2)
    .reset_index()
)
monthly_temp["mes"] = monthly_temp["month"].apply(
    lambda m: date(2024, m, 1).strftime("%B")
)
monthly_temp = monthly_temp.rename(columns={"temp_mean": "temp_media_C"})

print("Temperatura média mensal no Rio de Janeiro (jan–ago 2024):\n")
display(monthly_temp[["mes", "temp_media_C"]].reset_index(drop=True))

---
### Pergunta 5 — Qual foi o tempo predominante em cada mês?

In [ ]:
monthly_wcode = (
    weather.groupby("month")["weather_code"]
    .apply(lambda s: s.value_counts().idxmax())
    .reset_index()
)
monthly_wcode["mes"]       = monthly_wcode["month"].apply(
    lambda m: date(2024, m, 1).strftime("%B")
)
monthly_wcode["descricao"] = monthly_wcode["weather_code"].apply(wmo_desc)

print("Tempo predominante por mês (jan–ago 2024):\n")
display(monthly_wcode[["mes", "weather_code", "descricao"]].reset_index(drop=True))

---
### Pergunta 6 — Tempo e temperatura média em cada feriado (jan–ago 2024)?

In [ ]:
h_weather = holidays_period.merge(
    weather[["date", "temp_mean", "weather_code"]], on="date", how="left"
)
h_weather["descricao_tempo"] = h_weather["weather_code"].apply(wmo_desc)

print("Feriados de jan–ago 2024 com dados climáticos:\n")
display(
    h_weather[["date", "localName", "weekday", "temp_mean", "weather_code", "descricao_tempo"]]
    .reset_index(drop=True)
)

---
### Pergunta 7 — Houve algum feriado "não aproveitável" em 2024?

In [ ]:
# Critérios: feriado aproveitável = temp >= 20°C E tempo sem chuva/nublado total
h_weather["aproveitavel"] = h_weather.apply(is_beach_day, axis=1)

nao_aprov = h_weather[h_weather["aproveitavel"] == False]

print("Regras para feriado NÃO aproveitável:")
print("  • Temperatura média < 20°C (considerado 'frio' pelo carioca)")
print("  • OU tempo totalmente nublado, chuvoso ou com tempestade\n")

if len(nao_aprov) > 0:
    print(f"Feriados NÃO aproveitáveis em jan–ago 2024: {len(nao_aprov)}\n")
    display(
        nao_aprov[["date", "localName", "temp_mean", "weather_code", "descricao_tempo"]]
        .reset_index(drop=True)
    )
else:
    print("Nenhum feriado não aproveitável encontrado no período jan–ago 2024.")

# Feriados fora do período com dados
fora = holidays_all[holidays_all["date"] > pd.Timestamp(END_DATE)]
print(f"\n⚠️ Feriados de ago–dez 2024 ({len(fora)}) sem dados climáticos disponíveis:")
display(fora[["date", "localName", "weekday"]].reset_index(drop=True))

---
### Pergunta 8 — Qual foi o feriado "mais aproveitável" de 2024?

In [ ]:
aprov = h_weather[h_weather["aproveitavel"] == True].copy()

if len(aprov) > 0:
    # Score: prioriza menor weather_code (mais sol) + maior temperatura
    # Normaliza: score = temp_mean - weather_code * penalidade
    aprov["score"] = aprov["temp_mean"] - aprov["weather_code"] * 2
    aprov = aprov.sort_values("score", ascending=False)

    melhor = aprov.iloc[0]
    print("🏖️  Feriado MAIS APROVEITÁVEL de 2024:\n")
    print(f"  Data         : {melhor['date'].date()}")
    print(f"  Feriado      : {melhor['localName']}")
    print(f"  Dia da semana: {melhor['weekday']}")
    print(f"  Temperatura  : {melhor['temp_mean']}°C")
    print(f"  Tempo        : {melhor['descricao_tempo']} (código WMO {int(melhor['weather_code'])})")
    print("\nTodos os feriados aproveitáveis (ordenados por qualidade):\n")
    display(
        aprov[["date", "localName", "temp_mean", "weather_code", "descricao_tempo", "score"]]
        .reset_index(drop=True)
    )
else:
    print("Nenhum feriado aproveitável encontrado no período com dados climáticos.")